# Cooling Centers vs. 311 Requests

In [ ]:
import geopandas as gpd
import pandas as pd
import json
from shapely.geometry import shape
import shii
import os
from pointpats import distance_statistics as ds
from pointpats import PointPattern, random
from pointpats import f, g, k, l, g_test, k_test
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

APP_TOKEN = os.getenv('NYC_OPEN_DATA_APP_TOKEN')

## Load data

In [ ]:
def load_cooling_geojson(path):

    with open(path) as f:
        data = json.load(f)

    # Convert all field values to strings if they're time-typed (OGR type 10)
    for feature in data["features"]:
        props = feature["properties"]
        for key, val in props.items():
            # Time fields come through as "HH:MM:SS" strings in raw JSON anyway
            # so this is mostly a no-op, but forces fiona to skip its type inference
            if val is not None and not isinstance(val, (int, float, bool)):
                props[key] = str(val)

    gdf = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
    time_cols = [c for c in gdf.columns if "open" in c or "close" in c]
    for col in time_cols:
        gdf[col] = pd.to_datetime(gdf[col], format="%H:%M:%S", errors="coerce").dt.time
    return gdf

In [ ]:
cc_gdf = load_cooling_geojson('./data/cooling_centers.geojson').to_crs('EPSG:32618')
all_311_df = shii.prepare_all_311_requests(APP_TOKEN, output_path='./311_calls.gpkg').to_crs('EPSG:32618')
zc_gdf = shii.download_zipcodes().set_crs('EPSG:4326').to_crs('EPSG:32618')
boundary = zc_gdf.dissolve().to_crs('EPSG:32618')

In [ ]:
temp_311_gdf = all_311_df.loc[all_311_df['request_type']=='hydrant']
temp_cc_gdf = cc_gdf.loc[cc_gdf['Space_type'] == 'Cooling Center']
coords_311 = np.array(list(zip(temp_311_gdf.geometry.x, temp_311_gdf.geometry.y)))
coords_311 = coords_311[~np.max(np.isnan(coords_311), axis=1)]
cc_coords  = np.array(list(zip(temp_cc_gdf.geometry.x,  temp_cc_gdf.geometry.y)))

In [ ]:
ax= boundary.plot()
temp_cc_gdf.plot(ax=ax, color='red', markersize=0.75, alpha=0.7)

# Calculate Ripley's Cross-K

In [ ]:
# Ripley's cross-k correlation, clipped for boundary but NOT accounting for population density
def cross_k(pattern_a, pattern_b, distances, area=None, boundary=None):
    """
    Cross-K: for each point in A, count points in B within radius r.
    Returns observed K, simulation envelope (low, high), and theoretical K.
    """
    n_a = len(pattern_a)
    n_b = len(pattern_b)
    area = boundary.area.values[0]
    
    intensity_b = n_b / area

    tree = cKDTree(pattern_b)
    
    def k_observed(a, b_tree, dists):
        counts = np.array([b_tree.query_ball_point(a, r=d, return_length=True) 
                           for d in dists])  # shape: (n_dists, n_a)
        # counts is (n_dists, n_a) — mean over points in A
        return counts.mean(axis=1) / intensity_b

    observed = k_observed(pattern_a, tree, distances)
    
    rand_b_points = boundary.sample_points(n_b).explode()
    rand_b = np.column_stack([
        rand_b_points.geometry.x,
        rand_b_points.geometry.y
    ])
    sim = np.array((k_observed(pattern_a, cKDTree(rand_b), distances)))

    return observed, sim


# Run it
distances = np.linspace(0, 10000, 1000)

observed_k, sim = cross_k(cc_coords, coords_311, distances, boundary=boundary)

In [ ]:
# Theoretical K under CSR
theoretical_k = np.pi * distances**2

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(distances, observed_k, label="Observed K", color="steelblue", lw=2)
ax.plot(distances, theoretical_k, label="Theoretical (CSR)", color="black", 
        linestyle="--", lw=1.5)
ax.plot(distances, sim, alpha=0.2, color="gray", 
                label="Simulated k")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("K(d)")
ax.set_title("Cross-K Function: Cooling Centers vs. 311 Calls")
ax.legend()
plt.tight_layout()
plt.show()